# Пример 06. Решение системы методом исключения Гаусса

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** метод исключения Гаусса, общее решение системы $Ax = b$; связь с аналитическим обучением выходного слоя нейросети.

## Условие

Для системы $Ax = b$ с заданными $A\in\mathbb{R}^{3\times 6}$ и $b\in\mathbb{R}^3$ требуется найти общее решение методом исключения Гаусса и проверить его подстановкой.

## Математическая идея

Метод Гаусса состоит в приведении расширенной матрицы $[A\mid b]$ к ступенчатому виду с помощью элементарных преобразований строк:

- перестановка строк;
- умножение строки на ненулевое число;
- прибавление к одной строке другой, умноженной на число.

После приведения к ступенчатому виду свободные переменные переносятся в правую часть, а главные переменные выражаются через них. Общее решение имеет вид

$$x = x_p + \sum_{i=1}^{k} \alpha_i u_i,$$

где $x_p$ — частное решение, $u_i$ — базис нуль-пространства, $\alpha_i\in\mathbb{R}$.

## Решение

1. Матрица $A$ приводится к ступенчатому виду.
2. Определяются главные и свободные переменные.
3. Свободным переменным присваиваются параметры $s, t, r\in\mathbb{R}$.
4. Главные переменные выражаются через $s, t, r$.
5. Получается общее решение вида $x = (s,\, 2-r,\, t,\, -1-r,\, r,\, r-1)^\top$.

## Реализация на Python

Вектор $x$ подставляется в `A @ x` для конкретных значений $s=1$, $t=2$, $r=3$. Результат должен совпасть с $b = (2, -1, 1)^\top$. Во второй части рассматривается аналитическое решение задачи регрессии для выходного слоя нейросети: веса $w$ минимизируют $\|Hw - Y\|^2$, и оптимальное решение находится из нормальных уравнений

$$(H^\top H + \lambda I)\, w = H^\top Y,$$

которые решаются через `np.linalg.solve`. Малая регуляризация $\lambda = 10^{-5}$ добавляется для устойчивости.


In [4]:
import numpy as np

# Проверка общего решения
s, t, r = 1, 2, 3
x = np.array([s, 2 - r, t, -1 - r, r, r - 1])
A = np.array([[0, 1, 0, 0, 1, 0],
              [0, 0, 0, 1, 1, 0],
              [0, 1, 0, 0, 0, 1]])

b = np.array([2, -1, 1])

print("Ax =", A @ x)  # Должно быть [2, -1, 1]

Ax = [ 2 -1  1]


## Дополнительный пример

**Идея.** Если скрытые представления $H$ уже вычислены, то обучение выходного слоя линейной по параметрам сводится к решению нормальных уравнений. Это альтернатива итерационному градиентному спуску: вместо тысяч шагов — одно решение системы.

**Что демонстрирует код.** Генерируются случайные скрытые активации $H\in\mathbb{R}^{100\times 10}$ и цели $Y\in\mathbb{R}^{100\times 1}$. Матрица $A = H^\top H + \lambda I$ и вектор $b = H^\top Y$ подаются в `np.linalg.solve`, который внутренне использует LU-разложение — прямой потомок метода Гаусса.


### Метод Гаусса и обучение нейросетей: где они пересекаются? Аналитическое обучение выходного слоя

In [3]:
import numpy as np

# Скрытые активации (случайные)
np.random.seed(42)
H = np.random.randn(100, 10)  # 100 объектов, 10 признаков
Y = np.random.randn(100, 1)  # Целевой вектор

# Аналитическое решение (метод Гаусса через np.linalg.solve)
A = H.T @ H + 1e-5 * np.eye(10)  # Регуляризация
b = H.T @ Y
w_opt = np.linalg.solve(A, b)

print("Оптимальные веса:", w_opt.flatten()[:3])  # Первые 3 веса

Оптимальные веса: [-0.0889717  -0.05294987 -0.0323867 ]


## Проверка результата

В первой части проверка состоит в подстановке найденного общего решения в исходную систему для конкретных значений свободных параметров. Совпадение `A @ x` с вектором $b$ подтверждает корректность решения. Во второй части корректность аналитического решения гарантируется тем, что `np.linalg.solve` решает нормальные уравнения с малой регуляризацией; полученные веса $w$ минимизируют сумму квадратов невязок.

## Вывод

Метод Гаусса — исторически первый и до сих пор практический способ решения линейных систем. В машинном обучении его потомки (LU-разложение, QR-разложение) используются для аналитического обучения выходных слоёв, для вычисления матрицы ковариации и для оценки параметров линейных моделей. Понимание структуры общего решения помогает диагностировать переобучение и идентифицировать вырожденные задачи.
